# Customer Personality Analysis & Segmentation

## Objective
Analyze customer demographics, purchasing behavior, campaign responses, and channel activity to identify meaningful customer segments.

### Workflow
1. Load and inspect the raw data
2. Clean missing values, duplicates, categories, and dates
3. Engineer customer-level behavioral features
4. Perform exploratory data analysis
5. Build customer segments using K-Means clustering
6. Evaluate clustering with silhouette score
7. Interpret segments and derive business recommendations

## 1. Import Libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
sns.set_theme(style="whitegrid")

## 2. Load Dataset

In [ ]:
# Keep the CSV in the same folder as this notebook.
# Both the original filename and a cleaner filename are supported.

possible_files = [
    "marketing_campaign.csv",
    "marketing_campaign (1).csv"
]

data_file = next((f for f in possible_files if os.path.exists(f)), None)

if data_file is None:
    raise FileNotFoundError(
        "Dataset not found. Put 'marketing_campaign.csv' "
        "in the same folder as this notebook."
    )

df = pd.read_csv(data_file, sep="\t")

print(f"Loaded file: {data_file}")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
display(df.head())

## 3. Initial Data Quality Assessment

In [ ]:
print("Dataset shape:", df.shape)

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nMissing values:")
missing = df.isna().sum().sort_values(ascending=False)
display(missing[missing > 0].to_frame("missing_count"))

print("Duplicate rows:", df.duplicated().sum())
print("Duplicate customer IDs:", df["ID"].duplicated().sum())

## 4. Data Cleaning

In [ ]:
clean_df = df.copy()

# Remove exact duplicate rows.
clean_df = clean_df.drop_duplicates().copy()

# Each customer ID should represent one customer record.
clean_df = clean_df.drop_duplicates(subset="ID", keep="first").copy()

# Convert Income to numeric and impute missing values with the median.
clean_df["Income"] = pd.to_numeric(clean_df["Income"], errors="coerce")
income_median = clean_df["Income"].median()
clean_df["Income"] = clean_df["Income"].fillna(income_median)

# Clean categorical text.
for col in ["Education", "Marital_Status"]:
    clean_df[col] = clean_df[col].astype(str).str.strip()

# Consolidate uncommon marital-status labels.
marital_map = {
    "Alone": "Single",
    "Absurd": "Other",
    "YOLO": "Other"
}
clean_df["Marital_Status"] = clean_df["Marital_Status"].replace(marital_map)

# Consolidate the 2n Cycle education label.
clean_df["Education"] = clean_df["Education"].replace({
    "2n Cycle": "Master"
})

# Correctly parse DD-MM-YYYY dates.
clean_df["Dt_Customer"] = pd.to_datetime(
    clean_df["Dt_Customer"],
    format="%d-%m-%Y",
    errors="coerce"
)

print("Shape after cleaning:", clean_df.shape)
print("Remaining missing values:", int(clean_df.isna().sum().sum()))
print("Duplicate rows:", clean_df.duplicated().sum())
print("Duplicate IDs:", clean_df["ID"].duplicated().sum())

## 5. Data Validity Checks

In [ ]:
print(
    "Year_Birth range:",
    clean_df["Year_Birth"].min(),
    "to",
    clean_df["Year_Birth"].max()
)

# Remove implausible birth years before age-based analysis.
valid_birth_year = clean_df["Year_Birth"].between(1900, 2000)
print("Invalid birth-year records:", int((~valid_birth_year).sum()))

clean_df = clean_df.loc[valid_birth_year].copy()

# Use the latest customer date in the dataset as the reference year.
reference_year = clean_df["Dt_Customer"].dt.year.max()
clean_df["Age"] = reference_year - clean_df["Year_Birth"]

valid_age = clean_df["Age"].between(18, 100)
print("Invalid age records:", int((~valid_age).sum()))

clean_df = clean_df.loc[valid_age].copy()

print("Final analytical dataset shape:", clean_df.shape)

## 6. Feature Engineering

In [ ]:
spending_cols = [
    "MntWines",
    "MntFruits",
    "MntMeatProducts",
    "MntFishProducts",
    "MntSweetProducts",
    "MntGoldProds"
]

purchase_cols = [
    "NumWebPurchases",
    "NumCatalogPurchases",
    "NumStorePurchases"
]

campaign_cols = [
    "AcceptedCmp1",
    "AcceptedCmp2",
    "AcceptedCmp3",
    "AcceptedCmp4",
    "AcceptedCmp5",
    "Response"
]

clean_df["Total_Spend"] = clean_df[spending_cols].sum(axis=1)
clean_df["Total_Purchases"] = clean_df[purchase_cols].sum(axis=1)
clean_df["Total_Children"] = clean_df["Kidhome"] + clean_df["Teenhome"]
clean_df["Accepted_Campaigns"] = clean_df[campaign_cols].sum(axis=1)

clean_df["Customer_Tenure_Days"] = (
    clean_df["Dt_Customer"].max() - clean_df["Dt_Customer"]
).dt.days

clean_df["Customer_Tenure_Years"] = (
    clean_df["Customer_Tenure_Days"] / 365.25
)

feature_cols = [
    "Age",
    "Income",
    "Recency",
    "Total_Spend",
    "Total_Purchases",
    "Total_Children",
    "Accepted_Campaigns",
    "NumDealsPurchases",
    "NumWebVisitsMonth"
]

display(clean_df[feature_cols].describe().T)

## 7. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(clean_df["Income"], kde=True, ax=axes[0])
axes[0].set_title("Customer Income Distribution")
axes[0].set_xlabel("Income")

sns.histplot(clean_df["Total_Spend"], kde=True, ax=axes[1])
axes[1].set_title("Total Customer Spending")
axes[1].set_xlabel("Total Spend")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=clean_df,
    x="Income",
    y="Total_Spend",
    hue="Response",
    alpha=0.7
)

plt.title("Income vs Total Spending")
plt.xlabel("Income")
plt.ylabel("Total Spend")
plt.show()

In [ ]:
campaign_response_rate = clean_df["Response"].mean() * 100
print(f"Latest campaign response rate: {campaign_response_rate:.2f}%")

response_summary = (
    clean_df
    .groupby("Response")[["Income", "Total_Spend", "Total_Purchases", "Recency"]]
    .mean()
    .round(2)
)

display(response_summary)

## 8. Correlation Analysis

In [ ]:
corr_cols = [
    "Income",
    "Recency",
    "Total_Spend",
    "Total_Purchases",
    "Total_Children",
    "Accepted_Campaigns",
    "NumDealsPurchases",
    "NumWebVisitsMonth",
    "Response"
]

plt.figure(figsize=(11, 8))

sns.heatmap(
    clean_df[corr_cols].corr(),
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)

plt.title("Correlation Matrix of Key Customer Features")
plt.tight_layout()
plt.show()

## 9. Customer Segmentation with K-Means

In [ ]:
cluster_features = [
    "Income",
    "Recency",
    "Total_Spend",
    "Total_Purchases",
    "Total_Children",
    "Accepted_Campaigns",
    "NumDealsPurchases",
    "NumWebVisitsMonth"
]

X = clean_df[cluster_features].copy()

# Reduce skew in positive-valued variables.
log_features = [
    "Income",
    "Total_Spend",
    "Total_Purchases",
    "NumDealsPurchases"
]

for col in log_features:
    X[col] = np.log1p(X[col].clip(lower=0))

# Standardize before K-Means.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Clustering matrix shape:", X_scaled.shape)

In [ ]:
cluster_results = []

for k in range(2, 9):
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    labels = model.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)

    cluster_results.append({
        "k": k,
        "silhouette_score": score
    })

cluster_scores = pd.DataFrame(cluster_results)
display(cluster_scores)

best_k = int(
    cluster_scores.loc[
        cluster_scores["silhouette_score"].idxmax(),
        "k"
    ]
)

print("Best k based on silhouette score:", best_k)

In [ ]:
kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=10
)

clean_df["Cluster"] = kmeans.fit_predict(X_scaled)

final_silhouette = silhouette_score(
    X_scaled,
    clean_df["Cluster"]
)

print(f"Final silhouette score: {final_silhouette:.3f}")

print("\nCustomers per cluster:")
display(
    clean_df["Cluster"]
    .value_counts()
    .sort_index()
    .to_frame("customer_count")
)

## 10. Cluster Profiling

In [ ]:
cluster_profile = (
    clean_df
    .groupby("Cluster")
    .agg(
        Customers=("ID", "count"),
        Avg_Income=("Income", "mean"),
        Avg_Recency=("Recency", "mean"),
        Avg_Total_Spend=("Total_Spend", "mean"),
        Avg_Total_Purchases=("Total_Purchases", "mean"),
        Avg_Children=("Total_Children", "mean"),
        Avg_Campaign_Acceptance=("Accepted_Campaigns", "mean"),
        Avg_Web_Visits=("NumWebVisitsMonth", "mean"),
        Campaign_Response_Rate=("Response", "mean")
    )
    .round(2)
)

cluster_profile["Campaign_Response_Rate"] = (
    cluster_profile["Campaign_Response_Rate"] * 100
).round(2)

display(cluster_profile)

In [ ]:
plt.figure(figsize=(10, 6))

sns.barplot(
    data=cluster_profile.reset_index(),
    x="Cluster",
    y="Avg_Total_Spend"
)

plt.title("Average Customer Spending by Segment")
plt.xlabel("Customer Segment")
plt.ylabel("Average Total Spend")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=clean_df,
    x="Recency",
    y="Total_Spend",
    hue="Cluster",
    palette="tab10",
    s=70,
    alpha=0.75
)

plt.title("Customer Segments: Recency vs Total Spending")
plt.xlabel("Recency")
plt.ylabel("Total Spend")
plt.legend(title="Cluster")
plt.show()

## 11. Segment Interpretation

In [ ]:
segment_rank = cluster_profile.sort_values(
    "Avg_Total_Spend",
    ascending=False
)

display(
    segment_rank[
        [
            "Customers",
            "Avg_Income",
            "Avg_Recency",
            "Avg_Total_Spend",
            "Avg_Total_Purchases",
            "Avg_Campaign_Acceptance",
            "Campaign_Response_Rate"
        ]
    ]
)

print("""
Interpretation guide:
- High spend + high purchase frequency -> high-value customers.
- High recency -> customers who purchased less recently and may need re-engagement.
- High campaign response -> stronger candidates for targeted promotions.
- High web visits with lower purchases -> potential conversion opportunity.
- Higher deal usage -> potentially more price-sensitive behavior.
""")

## 12. Business Recommendations

In [ ]:
highest_spend_cluster = cluster_profile["Avg_Total_Spend"].idxmax()
lowest_spend_cluster = cluster_profile["Avg_Total_Spend"].idxmin()
best_response_cluster = cluster_profile["Campaign_Response_Rate"].idxmax()

print(f"Highest-spending segment: Cluster {highest_spend_cluster}")
print(f"Lowest-spending segment: Cluster {lowest_spend_cluster}")
print(f"Highest campaign-response segment: Cluster {best_response_cluster}")

print("\nRecommended actions:")
print("1. Prioritize high-value segments with retention and loyalty campaigns.")
print("2. Re-engage customers with high recency using targeted offers.")
print("3. Use campaign-response behavior to personalize marketing outreach.")
print("4. Investigate high web-visit / low-purchase customers for conversion improvements.")
print("5. Use segment-level behavior to guide product recommendations.")

## 13. Conclusion

This project transforms raw customer marketing data into actionable customer segments.

### Technical skills demonstrated
- Python
- Pandas and NumPy
- Data cleaning and validation
- Missing-value treatment
- Feature engineering
- Exploratory data analysis
- Data visualization
- Feature scaling
- K-Means clustering
- Silhouette-score evaluation
- Business interpretation

### Future improvements
- Compare K-Means with hierarchical clustering or DBSCAN
- Build a Power BI dashboard for segment exploration
- Train a campaign-response prediction model
- Create an API that predicts a customer's segment
- Add automated tests and reproducible project setup